In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the cleaned dataset (not the raw one — we already removed the duplicate)
df = pd.read_csv("../data/processed/heart_disease_clean.csv")
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


In [2]:
# Separate features (X) from target (y)
X = df.drop("target", axis=1)
y = df["target"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

# 80:20 stratified split — stratify=y ensures both train and test sets
# keep the same proportion of disease/no-disease cases as the full dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining set:", X_train.shape)
print("Testing set:", X_test.shape)

print("\nTraining set class balance:")
print(y_train.value_counts(normalize=True))
print("\nTesting set class balance:")
print(y_test.value_counts(normalize=True))

Features shape: (302, 13)
Target shape: (302,)

Training set: (241, 13)
Testing set: (61, 13)

Training set class balance:
target
1    0.543568
0    0.456432
Name: proportion, dtype: float64

Testing set class balance:
target
1    0.540984
0    0.459016
Name: proportion, dtype: float64


In [3]:
# Fit the scaler on TRAINING data only, then apply to both sets
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)

# Sanity check: scaled training data should have mean ~0, std ~1
import numpy as np
print("\nMean of scaled training features (should be ~0):")
print(np.round(X_train_scaled.mean(axis=0), 2))
print("\nStd of scaled training features (should be ~1):")
print(np.round(X_train_scaled.std(axis=0), 2))

X_train_scaled shape: (241, 13)
X_test_scaled shape: (61, 13)

Mean of scaled training features (should be ~0):
[ 0.  0. -0. -0.  0. -0. -0. -0. -0. -0.  0.  0. -0.]

Std of scaled training features (should be ~1):
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Initialize and train the model
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train_scaled, y_train)

# Predict on the unseen test set
y_pred = rf_model.predict(X_test_scaled)

# Evaluate
print("Accuracy: ", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:   ", recall_score(y_test, y_pred))
print("F1-score: ", f1_score(y_test, y_pred))

print("\nFull classification report:")
print(classification_report(y_test, y_pred))

Accuracy:  0.7868852459016393
Precision: 0.75
Recall:    0.9090909090909091
F1-score:  0.821917808219178

Full classification report:
              precision    recall  f1-score   support

           0       0.86      0.64      0.73        28
           1       0.75      0.91      0.82        33

    accuracy                           0.79        61
   macro avg       0.80      0.78      0.78        61
weighted avg       0.80      0.79      0.78        61



In [5]:
from sklearn.model_selection import cross_val_score

# 10-fold cross-validation, as specified in the methodology
cv_scores = cross_val_score(rf_model, X_train_scaled, y_train, cv=10, scoring="accuracy")

print("Accuracy for each of the 10 folds:")
print(np.round(cv_scores, 3))
print(f"\nMean CV Accuracy: {cv_scores.mean():.4f}")
print(f"Standard Deviation: {cv_scores.std():.4f}")

Accuracy for each of the 10 folds:
[0.84  0.667 0.833 0.833 0.875 0.708 0.917 0.833 0.875 0.875]

Mean CV Accuracy: 0.8257
Standard Deviation: 0.0742


In [6]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# Define all four models required by the methodology
models = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Support Vector Machine": SVC(random_state=42)
}

results = []

for name, model in models.items():
    # 10-fold cross-validation accuracy on training data
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=10, scoring="accuracy")

    # Also fit on full training set and evaluate on the held-out test set
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    results.append({
        "Model": name,
        "CV Mean Accuracy": cv_scores.mean(),
        "CV Std": cv_scores.std(),
        "Test Accuracy": accuracy_score(y_test, y_pred),
        "Test Precision": precision_score(y_test, y_pred),
        "Test Recall": recall_score(y_test, y_pred),
        "Test F1": f1_score(y_test, y_pred)
    })

results_df = pd.DataFrame(results).sort_values("CV Mean Accuracy", ascending=False)
results_df

,Model,CV Mean Accuracy,CV Std,Test Accuracy,Test Precision,Test Recall,Test F1
0,Random Forest,0.825667,0.074170,0.786885,0.750000,0.909091,0.821918
3,Support Vector Machine,0.821500,0.091495,0.836066,0.794872,0.939394,0.861111
2,K-Nearest Neighbors,0.804500,0.126750,0.803279,0.769231,0.909091,0.833333
1,Decision Tree,0.792667,0.066630,0.721311,0.710526,0.818182,0.760563


In [7]:
import joblib

# Save the best-performing model AND the scaler (both are needed for prediction later)
best_model = models["Support Vector Machine"]

joblib.dump(best_model, "../saved_models/heart_disease_model.pkl")
joblib.dump(scaler, "../saved_models/heart_disease_scaler.pkl")

print("Model and scaler saved successfully.")

Model and scaler saved successfully.


In [8]:
from sklearn.svm import SVC

# Retrain SVM specifically WITH probability estimation enabled
# (needed for proper low/moderate/high risk stratification)
best_model = SVC(random_state=42, probability=True)
best_model.fit(X_train_scaled, y_train)

y_pred = best_model.predict(X_test_scaled)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Recall:  ", recall_score(y_test, y_pred))

# Re-save the model with probability support
joblib.dump(best_model, "../saved_models/heart_disease_model.pkl")
print("Model re-saved with probability estimation enabled.")

Accuracy: 0.8360655737704918
Recall:   0.9393939393939394
Model re-saved with probability estimation enabled.


D:\ai-disease-detection\venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [9]:
# Grab a few real patients from the test set with their TRUE labels,
# so we can verify the Flask API against ground truth
sample = X_test.iloc[0]
true_label = y_test.iloc[0]

print("True label (0 = no disease, 1 = disease):", true_label)
print("\nFeatures as JSON for curl:")
print(sample.to_dict())

True label (0 = no disease, 1 = disease): 0

Features as JSON for curl:
{'age': 57.0, 'sex': 1.0, 'cp': 0.0, 'trestbps': 150.0, 'chol': 276.0, 'fbs': 0.0, 'restecg': 0.0, 'thalach': 112.0, 'exang': 1.0, 'oldpeak': 0.6, 'slope': 1.0, 'ca': 1.0, 'thal': 1.0}


In [10]:
# Find one confirmed disease-positive and one confirmed disease-negative case
positive_idx = y_test[y_test == 1].index[0]
negative_idx = y_test[y_test == 0].index[0]

print("=== CONFIRMED POSITIVE CASE (target=1) ===")
print(X_test.loc[positive_idx].to_dict())

print("\n=== CONFIRMED NEGATIVE CASE (target=0) ===")
print(X_test.loc[negative_idx].to_dict())

=== CONFIRMED POSITIVE CASE (target=1) ===
{'age': 51.0, 'sex': 0.0, 'cp': 2.0, 'trestbps': 120.0, 'chol': 295.0, 'fbs': 0.0, 'restecg': 0.0, 'thalach': 157.0, 'exang': 0.0, 'oldpeak': 0.6, 'slope': 2.0, 'ca': 0.0, 'thal': 2.0}

=== CONFIRMED NEGATIVE CASE (target=0) ===
{'age': 57.0, 'sex': 1.0, 'cp': 0.0, 'trestbps': 150.0, 'chol': 276.0, 'fbs': 0.0, 'restecg': 0.0, 'thalach': 112.0, 'exang': 1.0, 'oldpeak': 0.6, 'slope': 1.0, 'ca': 1.0, 'thal': 1.0}
